In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

集合クエリ（Set Query）は、`2つ以上のSELECT文の結果`を1つの結果セットにまとめるクエリである。

各`SELECT文`は、次の条件を満たす必要がある。

- カラム数が同じであること
- 対応するカラムのデータ型に互換性があること<br>(カラム名が異なっていても、データ型に互換性があれば演算できる)
- 結果セットのカラム名には、最初の`SELECT文`のカラム名が使用される

| 集合演算子 | 意味 | 
|---|---|
| `UNION` | 重複を削除して結合する | 
| `UNION ALL` | 重複を含めて結合する | 
| `INTERSECT` | 共通するデータのみを取得する | 
| `EXCEPT` | 1つ目の結果から2つ目の結果を除外する | 

## 1. UNION

- `UNION`は、複数の`SELECT文`の結果を結合し、重複する行を削除する。

次の例では、俳優の名前と顧客の名前を1つの結果として取得する。

In [4]:
%%sql

SELECT first_name
FROM actor

UNION

SELECT first_name
FROM customer

LIMIT 6;

first_name
PENELOPE
NICK
ED
JENNIFER
JOHNNY
BETTE


## 2. UNION ALL

- `UNION ALL`は、重複する行を削除せず、すべての結果を結合する。

次の例では、俳優の名前と顧客の名前を重複も含めて1つの結果として取得する。

In [8]:
%%sql

SELECT first_name
FROM actor

UNION ALL

SELECT first_name
FROM customer

ORDER BY first_name -- 重複を確認するため、`UNION ALL`で結合した後に並べ替える。

LIMIT 5;

first_name
AARON
ADAM
ADAM
ADAM
ADRIAN


## 3. INTERSECT

- `INTERSE`は、2つの検索結果の両方に存在するデータのみを取得する。

次の例では、俳優と顧客の両方に存在する名前のみが表示される。

In [11]:
%%sql

SELECT first_name
FROM actor

INTERSECT

SELECT first_name
FROM customer

LIMIT 5;

first_name
JENNIFER
JOHNNY
GRACE
MATTHEW
JOE


## 4. EXCEPT

- `EXCEPT`は、1つ目の検索結果には存在するが、2つ目の検索結果には存在しないデータのみを取得する。

次の例では、俳優の名前のうち、顧客の名前には存在しないデータのみが表示される。

In [12]:
%%sql

SELECT first_name
FROM actor

EXCEPT

SELECT first_name
FROM customer

LIMIT 5;

first_name
PENELOPE
NICK
ED
BETTE
ZERO


#### ※ カラム名が異なる場合

- カラム名が異なっていても、カラム数とデータ型に互換性があれば結合できる。

In [14]:
%%sql

SELECT first_name
FROM actor

UNION

SELECT city
FROM city

LIMIT 5;

first_name
PENELOPE
NICK
ED
JENNIFER
JOHNNY


#### ※ カラム数が異なる場合

- カラム数が異なるとエラーが発生する。

In [15]:
%%sql

SELECT first_name, last_name
FROM actor

UNION

SELECT first_name
FROM customer;

RuntimeError: (pymysql.err.OperationalError) (1222, 'The used SELECT statements have a different number of columns')
[SQL: SELECT first_name, last_name
FROM actor

UNION

SELECT first_name
FROM customer;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### データ型が異なる場合

- データ型が異なる場合、MySQLは可能な範囲で自動的に型変換を行う。

In [17]:
%%sql

SELECT actor_id
FROM actor

UNION

SELECT first_name
FROM customer

LIMIT 5;

actor_id
58
92
182
118
145
